# SBERT — Table 3: Argument Facet Similarity (AFS)

This notebook reproduces the **SBERT-AFS-base** experiment from Table 3 of *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*.

### Paper setup

- Backbone: `bert-base-uncased`
- SBERT: shared/tied BERT encoders
- Pooling: MEAN
- Objective: regression
- Similarity: cosine similarity
- Loss: mean squared error
- Dataset: Argument Facet Similarity (AFS)
- Topics: Death Penalty, Gun Control, Gay Marriage
- Evaluation:
  1. 10-fold cross-validation
  2. Cross-topic evaluation

The paper reports Pearson and Spearman correlation for both evaluation settings.

**Important:** This notebook keeps the two evaluation protocols separate. The 10-fold experiment trains a fresh model for each fold; the cross-topic experiment trains a fresh model for each held-out topic.


In [19]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


PyTorch: 2.11.0+cu128
CUDA available: True
Device: cuda


In [20]:
# Reproducibility

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [21]:
# Paths and training configuration

MODEL_NAME = "bert-base-uncased"

DATA_DIR = Path("..") / "data"
AFS_DIR = DATA_DIR / "AFS"

OUTPUT_DIR = Path("outputs") / "sbert_afs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PAIR_FILES = {
    "death_penalty": AFS_DIR / "ArgPairs_DP.csv",
    "gun_control": AFS_DIR / "ArgPairs_GC.csv",
    "gay_marriage": AFS_DIR / "ArgPairs_GM.csv",
}

MAX_LENGTH = 128

# Practical configuration for the RTX 3050 6GB.
PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = 16

EPOCHS = 4  # paper uses 4 epochs for STSb; AFS epochs not reported
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1   # paper: linear warm-up over 10% of training data

# Table 3 "SBERT-AFS-base" = plain bert-base-uncased fine-tuned on AFS only.
# Set True only if you deliberately want the NLI->AFS variant (NOT in the paper).
INIT_FROM_NLI = False

EVAL_BATCH_SIZE = 32
USE_AMP = torch.cuda.is_available()

print("AFS directory:", AFS_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


AFS directory: F:\SEM-7\NLP\Project\data\AFS
Output directory: F:\SEM-7\NLP\Project\notebooks\outputs\sbert_afs


In [22]:
# Load and combine the three AFS topic files

parts = []

for topic, path in PAIR_FILES.items():
    df = pd.read_csv(path, encoding="latin1")

    temp = df[["sentence_1", "sentence_2", "regression_label"]].copy()
    temp["topic"] = topic

    parts.append(temp)

afs = pd.concat(parts, ignore_index=True)

afs["sentence_1"] = afs["sentence_1"].astype(str)
afs["sentence_2"] = afs["sentence_2"].astype(str)
afs["regression_label"] = (
    pd.to_numeric(
        afs["regression_label"],
        errors="coerce"
    ) / 5.0
)

print("Shape:", afs.shape)
print("\nPairs by topic:")
print(afs["topic"].value_counts())

print("\nMissing values:")
print(afs.isna().sum())

print("\nScore range:")
print(afs["regression_label"].min(), "to", afs["regression_label"].max())

display(afs.head())


Shape: (6000, 4)

Pairs by topic:
topic
death_penalty    2000
gun_control      2000
gay_marriage     2000
Name: count, dtype: int64

Missing values:
sentence_1          0
sentence_2          0
regression_label    0
topic               0
dtype: int64

Score range:
0.06666666660000001 to 1.0


,sentence_1,sentence_2,regression_label,topic
0,Yes there should be a death penalty but it sho...,If a person just raped someone (this is just a...,0.866667,death_penalty
1,"In every state that retains the death penalty,...",I am advocating life in prison without the opt...,0.333333,death_penalty
2,so you weigh whatever monetary savings (if any...,You need to find lots more for the death penal...,0.466667,death_penalty
3,I am very conservative and I am opposed to abo...,I am not a big proponent of the death penalty.,0.733333,death_penalty
4,If it has nothing to do with the death penalty...,"(If a murderer is released and murders again, ...",0.600000,death_penalty


## SBERT-AFS regression model

For each sentence:

```text
Sentence → BERT → MEAN pooling → sentence embedding
```

For a pair:

```text
u, v → cosine(u, v) → predicted similarity
```

The regression objective compares the cosine similarity with the AFS gold similarity score using MSE.


In [23]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AFSRegressionDataset(Dataset):
    def __init__(self, dataframe):
        self.sentence1 = dataframe["sentence_1"].tolist()
        self.sentence2 = dataframe["sentence_2"].tolist()
        self.scores = dataframe["regression_label"].astype(float).tolist()

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return (
            self.sentence1[idx],
            self.sentence2[idx],
            self.scores[idx]
        )


def collate_fn(batch):
    sentences1 = [item[0] for item in batch]
    sentences2 = [item[1] for item in batch]
    scores = torch.tensor(
        [item[2] for item in batch],
        dtype=torch.float32
    )

    inputs1 = tokenizer(
        sentences1,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    inputs2 = tokenizer(
        sentences2,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return inputs1, inputs2, scores


In [24]:
NLI_MODEL_PATH = Path("outputs") / "sbert_nli_base" / "model" / "sbert_nli_base.pt"

class SBERTAFSRegressor(nn.Module):
    def __init__(self, model_name, checkpoint_path=None):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)

        if checkpoint_path is not None:
            checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
            state = checkpoint["model_state_dict"]
            bert_state = {k[len("bert."):]: v for k, v in state.items() if k.startswith("bert.")}
            assert len(bert_state) > 0, "No 'bert.' keys found in checkpoint - nothing would be loaded!"
            missing, unexpected = self.bert.load_state_dict(bert_state, strict=False)
            print(f"Loaded NLI weights | missing: {len(missing)} | unexpected: {len(unexpected)}")

    @staticmethod
    def mean_pooling(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def encode(self, inputs):
        outputs = self.bert(**inputs)
        return self.mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])

    def forward(self, inputs1, inputs2):
        u = self.encode(inputs1)
        v = self.encode(inputs2)
        # Raw cosine similarity (Fig. 2 of the paper) - compared directly with the
        # gold score scaled to [0,1]. NO (cos+1)/2 remapping.
        return torch.nn.functional.cosine_similarity(u, v, dim=1)


In [25]:
def make_loader(dataframe, shuffle):
    dataset = AFSRegressionDataset(dataframe)

    return DataLoader(
        dataset,
        batch_size=PHYSICAL_BATCH_SIZE if shuffle else EVAL_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available()
    )


def evaluate_model(model, dataframe):
    model.eval()

    loader = make_loader(dataframe, shuffle=False)

    predictions = []
    gold_scores = []

    with torch.no_grad():
        for inputs1, inputs2, scores in loader:
            inputs1 = {
                key: value.to(DEVICE)
                for key, value in inputs1.items()
            }
            inputs2 = {
                key: value.to(DEVICE)
                for key, value in inputs2.items()
            }
            scores = scores.to(DEVICE)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                pred = model(inputs1, inputs2)

            predictions.extend(pred.detach().cpu().numpy())
            gold_scores.extend(scores.cpu().numpy())

    predictions = np.asarray(predictions)
    gold_scores = np.asarray(gold_scores)

    pearson = pearsonr(predictions, gold_scores).statistic
    spearman = spearmanr(predictions, gold_scores).statistic

    return {
        "pearson": pearson * 100,
        "spearman": spearman * 100
    }


In [26]:
from transformers import get_linear_schedule_with_warmup

def train_model(train_df, eval_df, seed, epochs=EPOCHS):
    set_seed(seed)

    model = SBERTAFSRegressor(
        MODEL_NAME,
        NLI_MODEL_PATH if INIT_FROM_NLI else None
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    train_loader = make_loader(train_df, shuffle=True)

    steps_per_epoch = (len(train_loader) + GRADIENT_ACCUMULATION_STEPS - 1) // GRADIENT_ACCUMULATION_STEPS
    total_steps = steps_per_epoch * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, (inputs1, inputs2, scores) in enumerate(train_loader):
            inputs1 = {k: v.to(DEVICE) for k, v in inputs1.items()}
            inputs2 = {k: v.to(DEVICE) for k, v in inputs2.items()}
            scores = scores.to(DEVICE)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                predictions = model(inputs1, inputs2)

            # loss in fp32, outside autocast
            loss = criterion(predictions.float(), scores)
            running_loss += loss.item()
            loss = loss / GRADIENT_ACCUMULATION_STEPS

            scaler.scale(loss).backward()

            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        metrics = evaluate_model(model, eval_df)  # logging only, no model selection
        epoch_loss = running_loss / len(train_loader)
        history.append({"epoch": epoch + 1, "train_loss": epoch_loss,
                        "eval_pearson": metrics["pearson"], "eval_spearman": metrics["spearman"]})
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {epoch_loss:.4f} | "
              f"Pearson: {metrics['pearson']:.4f} | Spearman: {metrics['spearman']:.4f}")

    return model, pd.DataFrame(history)


## 10-fold cross-validation

The paper's first AFS evaluation uses **10-fold cross-validation**.

For every fold:

- 90% of the AFS data is used for training.
- 10% is held out for evaluation.
- A fresh `bert-base-uncased` SBERT model is initialized.
- The model is trained with the regression objective.
- Pearson and Spearman correlations are calculated on the held-out fold.

The paper reports the average correlation across the 10 folds. fileciteturn15file0L343-L357


In [27]:
from sklearn.model_selection import KFold

N_FOLDS = 10

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

cv_results = []
cv_histories = []

for fold, (train_idx, test_idx) in enumerate(
    kf.split(afs),
    start=1
):
    print("\n" + "=" * 80)
    print(f"FOLD {fold}/{N_FOLDS}")
    print("=" * 80)

    train_df = afs.iloc[train_idx].reset_index(drop=True)
    test_df = afs.iloc[test_idx].reset_index(drop=True)

    model, history = train_model(
        train_df=train_df,
        eval_df=test_df,
        seed=SEED + fold - 1
    )

    metrics = evaluate_model(model, test_df)

    cv_results.append({
        "fold": fold,
        "pearson": metrics["pearson"],
        "spearman": metrics["spearman"]
    })

    history["fold"] = fold
    cv_histories.append(history)

    print(
        f"Fold {fold}: "
        f"Pearson={metrics['pearson']:.4f}, "
        f"Spearman={metrics['spearman']:.4f}"
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    


FOLD 1/10
Epoch 1/4 | Loss: 0.0416 | Pearson: 69.7996 | Spearman: 69.2460
Epoch 2/4 | Loss: 0.0141 | Pearson: 73.9225 | Spearman: 72.3037
Epoch 3/4 | Loss: 0.0081 | Pearson: 75.4789 | Spearman: 73.7030
Epoch 4/4 | Loss: 0.0056 | Pearson: 75.7650 | Spearman: 73.9021
Fold 1: Pearson=75.7650, Spearman=73.9021

FOLD 2/10
Epoch 1/4 | Loss: 0.0421 | Pearson: 71.9771 | Spearman: 69.7570
Epoch 2/4 | Loss: 0.0139 | Pearson: 76.3967 | Spearman: 73.8528
Epoch 3/4 | Loss: 0.0081 | Pearson: 77.0806 | Spearman: 74.3627
Epoch 4/4 | Loss: 0.0056 | Pearson: 77.3092 | Spearman: 74.5579
Fold 2: Pearson=77.3092, Spearman=74.5579

FOLD 3/10
Epoch 1/4 | Loss: 0.0419 | Pearson: 72.3091 | Spearman: 69.7616
Epoch 2/4 | Loss: 0.0143 | Pearson: 75.2056 | Spearman: 72.4453
Epoch 3/4 | Loss: 0.0081 | Pearson: 75.5778 | Spearman: 72.1826
Epoch 4/4 | Loss: 0.0056 | Pearson: 76.0645 | Spearman: 72.6128
Fold 3: Pearson=76.0645, Spearman=72.6128

FOLD 4/10
Epoch 1/4 | Loss: 0.0419 | Pearson: 72.7135 | Spearman: 70.490

In [28]:
cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

cv_summary = pd.DataFrame({
    "metric": ["Pearson", "Spearman"],
    "mean": [
        cv_results_df["pearson"].mean(),
        cv_results_df["spearman"].mean()
    ],
    "std": [
        cv_results_df["pearson"].std(),
        cv_results_df["spearman"].std()
    ]
})

display(cv_summary)


,fold,pearson,spearman
0,1,75.764964,73.902103
1,2,77.309159,74.557891
2,3,76.064529,72.612828
3,4,77.456202,74.415604
4,5,78.751784,75.791955
5,6,72.867518,70.729107
6,7,77.438501,76.221085
7,8,74.057530,71.631340
8,9,75.674677,72.339987
9,10,73.328793,69.857045


,metric,mean,std
0,Pearson,75.871366,1.949267
1,Spearman,73.205895,2.120930


## Cross-topic evaluation

The paper also evaluates generalization to an unseen topic.

For each topic:

```text
Train: two topics
Test:  remaining topic
```

The three splits are:

```text
Death Penalty + Gun Control → Gay Marriage
Death Penalty + Gay Marriage → Gun Control
Gun Control + Gay Marriage → Death Penalty
```

The paper averages the three held-out-topic results. fileciteturn15file0L343-L347


In [29]:
topics = sorted(afs["topic"].unique())

cross_topic_results = []

for held_out_topic in topics:
    print("\n" + "=" * 80)
    print("HELD-OUT TOPIC:", held_out_topic)
    print("=" * 80)

    train_df = afs[
        afs["topic"] != held_out_topic
    ].reset_index(drop=True)

    test_df = afs[
        afs["topic"] == held_out_topic
    ].reset_index(drop=True)

    model, history = train_model(
        train_df=train_df,
        eval_df=test_df,
        seed=SEED
    )

    metrics = evaluate_model(model, test_df)

    cross_topic_results.append({
        "held_out_topic": held_out_topic,
        "pearson": metrics["pearson"],
        "spearman": metrics["spearman"]
    })

    print(
        f"Pearson={metrics['pearson']:.4f}, "
        f"Spearman={metrics['spearman']:.4f}"
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



HELD-OUT TOPIC: death_penalty
Epoch 1/4 | Loss: 0.0427 | Pearson: 50.7347 | Spearman: 49.6871
Epoch 2/4 | Loss: 0.0124 | Pearson: 51.2472 | Spearman: 50.4932
Epoch 3/4 | Loss: 0.0071 | Pearson: 51.2807 | Spearman: 50.5692
Epoch 4/4 | Loss: 0.0048 | Pearson: 51.5595 | Spearman: 50.8345
Pearson=51.5595, Spearman=50.8345

HELD-OUT TOPIC: gay_marriage
Epoch 1/4 | Loss: 0.0445 | Pearson: 32.9602 | Spearman: 29.0845
Epoch 2/4 | Loss: 0.0150 | Pearson: 36.2616 | Spearman: 32.8103
Epoch 3/4 | Loss: 0.0087 | Pearson: 35.8553 | Spearman: 32.2719
Epoch 4/4 | Loss: 0.0060 | Pearson: 36.2759 | Spearman: 32.6152
Pearson=36.2759, Spearman=32.6152

HELD-OUT TOPIC: gun_control
Epoch 1/4 | Loss: 0.0469 | Pearson: 64.2948 | Spearman: 63.3432
Epoch 2/4 | Loss: 0.0159 | Pearson: 65.5124 | Spearman: 63.7436
Epoch 3/4 | Loss: 0.0091 | Pearson: 65.3844 | Spearman: 63.8761
Epoch 4/4 | Loss: 0.0067 | Pearson: 65.8013 | Spearman: 64.1580
Pearson=65.8013, Spearman=64.1580


In [30]:
cross_topic_df = pd.DataFrame(cross_topic_results)

display(cross_topic_df)

print(
    "Mean Pearson:",
    cross_topic_df["pearson"].mean()
)

print(
    "Mean Spearman:",
    cross_topic_df["spearman"].mean()
)


,held_out_topic,pearson,spearman
0,death_penalty,51.559500,50.834460
1,gay_marriage,36.275879,32.615200
2,gun_control,65.801347,64.158003


Mean Pearson: 51.21224185921926
Mean Spearman: 49.202554140641496


## Comparison with Table 3

Paper values for **SBERT-AFS-base**:

| Evaluation | Pearson | Spearman |
|---|---:|---:|
| 10-fold CV | 76.57 | 74.13 |
| Cross-topic | 52.34 | 50.65 |

The paper reports these as average correlations.



In [31]:
paper_table3 = pd.DataFrame({
    "evaluation": [
        "10-fold CV",
        "Cross-topic"
    ],
    "paper_pearson": [
        76.57,
        52.34
    ],
    "paper_spearman": [
        74.13,
        50.65
    ],
    "our_pearson": [
        cv_results_df["pearson"].mean(),
        cross_topic_df["pearson"].mean()
    ],
    "our_spearman": [
        cv_results_df["spearman"].mean(),
        cross_topic_df["spearman"].mean()
    ]
})

paper_table3["pearson_difference"] = (
    paper_table3["our_pearson"]
    - paper_table3["paper_pearson"]
)

paper_table3["spearman_difference"] = (
    paper_table3["our_spearman"]
    - paper_table3["paper_spearman"]
)

display(paper_table3)


,evaluation,paper_pearson,paper_spearman,our_pearson,our_spearman,pearson_difference,spearman_difference
0,10-fold CV,76.57,74.13,75.871366,73.205895,-0.698634,-0.924105
1,Cross-topic,52.34,50.65,51.212242,49.202554,-1.127758,-1.447446


In [32]:
# Save results

cv_results_df.to_csv(
    OUTPUT_DIR / "afs_10fold_results.csv",
    index=False
)

cv_summary.to_csv(
    OUTPUT_DIR / "afs_10fold_summary.csv",
    index=False
)

cross_topic_df.to_csv(
    OUTPUT_DIR / "afs_cross_topic_results.csv",
    index=False
)

paper_table3.to_csv(
    OUTPUT_DIR / "table3_comparison.csv",
    index=False
)

print("Results saved to:")
print(OUTPUT_DIR.resolve())


Results saved to:
F:\SEM-7\NLP\Project\notebooks\outputs\sbert_afs


## Reproduction note

This implementation uses the same core SBERT architecture and AFS evaluation protocols described in the paper.

Practical adaptations for the available GPU include:

- physical batch size 4
- gradient accumulation to obtain an effective batch size of 16
- maximum sequence length 128
- mixed-precision training

These hardware adaptations are implementation choices and should not be interpreted as additional hyperparameters reported by the 2019 paper.

The paper reports that SBERT-AFS-base was evaluated using 10-fold cross-validation and cross-topic evaluation, with cosine similarity used for the similarity score.
